In [1]:
# ============================================================
# Cell 0: GPU Check
# ============================================================
!nvidia-smi

Sat Mar  7 06:23:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# ============================================================
# Cell 1: Setup — Clone, install, GPU auto-detect, tier config
# ============================================================
import os, subprocess, sys, time, math

t0 = time.time()

# ── Clone repo ──
REPO = "/content/nst"
if not os.path.exists(REPO):
    print("Cloning repo...")
    subprocess.run(["git", "clone",
        "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git",
        REPO], check=True)
    print("Repo cloned")
else:
    r = subprocess.run(["git", "pull", "--ff-only"],
                       capture_output=True, text=True, cwd=REPO)
    print(f"git pull: {r.stdout.strip()}")

os.chdir(REPO)
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

# ── Install deps (keep Colab's pre-installed PyTorch + CUDA) ──
print("\nInstalling dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.46.3", "datasets==2.21.0",
    "sentencepiece>=0.1.99", "protobuf>=4.0",
    "pyyaml", "scikit-learn", "rank-bm25==0.2.2",
    "accelerate", "peft>=0.7.0", "tiktoken",
    "fsspec>=2023.6,<2025", "huggingface_hub>=0.21,<1.0",
    "scipy>=1.9.0"],
    check=True, capture_output=True, text=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
    capture_output=True, text=True)
print("pip done")

# ── Verify installs ──
import torch, transformers, datasets
print(f"\nPyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")

# ════════════════════════════════════════════════════════════════
#  GPU Auto-Detection & Tier-Based Configuration
#  Matches training/model_setup.py detect_gpu_tier() logic
# ════════════════════════════════════════════════════════════════
GPU_OVERRIDES = {}
SMOKE_OVERRIDES = {}
VERI_OVERRIDES = {}
USE_LARGE_MODEL = False

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = torch.cuda.get_device_name(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    vram_gb = vram / 1e9
    cc = (props.major, props.minor)

    print(f"\nGPU          : {gpu_name}")
    print(f"VRAM         : {vram_gb:.1f} GB")
    print(f"Compute Cap  : {cc[0]}.{cc[1]}")

    supports_bf16  = cc >= (8, 0)
    supports_tf32  = cc >= (8, 0)
    can_compile    = hasattr(torch, "compile")

    # ── Tier-based config (matches model_setup.py) ──
    if vram_gb >= 70:        # H100 80GB / A100 80GB
        TIER = "H100 / A100-80GB"
        BS, GA, WORKERS = 64, 1, 4
        USE_LARGE_MODEL = True
    elif vram_gb >= 35:      # A100 40GB
        TIER = "A100-40GB"
        BS, GA, WORKERS = 48, 1, 4
        USE_LARGE_MODEL = True
    elif vram_gb >= 20:      # L4 24GB
        TIER = "L4-24GB"
        BS, GA, WORKERS = 32, 1, 2
        USE_LARGE_MODEL = True
    else:                    # T4 16GB / other
        TIER = "T4-16GB"
        BS, GA, WORKERS = 16, 2, 0
        USE_LARGE_MODEL = False

    will_compile = can_compile and vram_gb >= 20

    # Common GPU overrides for ALL modes
    _common = {
        "train": {
            "batch_size":      BS,
            "grad_accum_steps": GA,
            "bf16":            supports_bf16,
            "fp16":            not supports_bf16,
            "torch_compile":   will_compile,
            "tf32":            supports_tf32,
            "benchmark":       True,
            "num_workers":     WORKERS,
        }
    }
    GPU_OVERRIDES = dict(_common)

    # Smoke: smaller batch, no compile
    SMOKE_OVERRIDES = {
        "train": {
            "batch_size":      min(BS, 32),
            "grad_accum_steps": 1,
            "bf16":            supports_bf16,
            "fp16":            not supports_bf16,
            "tf32":            supports_tf32,
            "num_workers":     WORKERS,
        }
    }

    # NST-VERI: A100/H100 gets full large model + LoRA
    VERI_OVERRIDES = dict(_common)
    if USE_LARGE_MODEL:
        VERI_OVERRIDES["model"] = {
            "name": "microsoft/deberta-v3-large",
            "use_lora": True,
            "lora_rank": 16,
            "lora_alpha": 32,
            "gradient_checkpointing": True,
        }
    else:
        VERI_OVERRIDES["model"] = {
            "name": "microsoft/deberta-v3-base",
            "use_lora": True,
            "lora_rank": 8,
            "lora_alpha": 16,
            "gradient_checkpointing": True,
        }

    eff_bs = BS * GA
    prec = "BF16" if supports_bf16 else "FP16"
    comp = "ON" if will_compile else "OFF"
    model_size = "LARGE (304M)" if USE_LARGE_MODEL else "BASE (86M)"

    # Enable GPU optimizations immediately
    if supports_tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    os.environ.setdefault("ATTN_IMPL", "sdpa")

    print(f"\n{'='*60}")
    print(f"  AUTO-CONFIGURED FOR: {TIER}")
    print(f"  Model       : DeBERTa-v3 {model_size}")
    print(f"  Batch size  : {BS} x {GA} = {eff_bs} effective")
    print(f"  Precision   : {prec}")
    print(f"  torch.compile: {comp}")
    print(f"  TF32        : {'ON' if supports_tf32 else 'OFF'}")
    print(f"  SDPA/Flash  : ON")
    print(f"  DataLoaders : {WORKERS} workers")
    print(f"{'='*60}")

    # Runtime estimate
    base_min = 80
    if   vram_gb >= 70:  speedup = 10.0
    elif vram_gb >= 35:  speedup = 7.0
    elif vram_gb >= 20:  speedup = 4.0
    else:                speedup = 1.0
    est_veri = base_min / speedup * 1.3  # VERI ~30% more work than neural
    est_all = est_veri + base_min/speedup + 15  # VERI + neural + overhead
    print(f"\n  Est. NST-VERI  : ~{est_veri:.0f} min")
    print(f"  Est. total     : ~{est_all:.0f} min")
else:
    print("\n  WARNING: No CUDA GPU detected!")
    print("  Go to: Runtime > Change runtime type > select GPU")

print(f"\nSetup complete ({time.time()-t0:.0f}s)")

: 

In [4]:
# ============================================================
# Cell 2: Build FEVER wiki cache (one-time, ~4 min)
# ============================================================
import os, sys, time

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_wiki_cache import WikiCache

cache_path = "data/fever_wiki.db"
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    print(f"Wiki cache already exists: {n} pages, "
          f"{os.path.getsize(cache_path)/1024/1024:.1f} MB")
    cache.close()
else:
    print("Building wiki cache from HuggingFace FEVER dataset...")
    print("Downloads ~300MB, takes ~4 minutes. One-time only.")
    t0 = time.time()
    import subprocess
    r = subprocess.run([sys.executable, "main.py", "build-fever-wiki-cache"],
                       capture_output=True, text=True, timeout=600)
    print(r.stdout[-2000:] if r.stdout else "(no stdout)")
    if r.returncode != 0:
        print(f"STDERR: {r.stderr[-1000:]}")
    else:
        print(f"\nWiki cache built in {time.time()-t0:.0f}s")

# Verify
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    sample = cache.titles()[:3]
    for t in sample:
        sents = cache.lookup(t)
        print(f"  '{t}': {len(sents)} sentences")
    cache.close()
    print(f"Wiki cache verified: {n} pages")
else:
    print("ERROR: Wiki cache not found!")

Building wiki cache from HuggingFace FEVER dataset...
This downloads ~300MB and takes ~4 minutes. One-time only.

  Built: 14363/14533 pages (170 missing) in 363.6s
  Cache: data/fever_wiki.db (24.2 MB)


Wiki cache built in 365s
  '"Heroes"_-LRB-David_Bowie_album-RRB-': 9 sentences
  ''Til_Death': 4 sentences
  '...More_Unchartered_Heights_of_Disgrace': 13 sentences
Wiki cache verified: 14363 pages


In [5]:
# ============================================================
# Cell 3: Smoke Test — 200 examples, 1 epoch (pipeline validation)
# ============================================================
import os, sys, time, logging

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  SMOKE TEST: 200 train, 100 dev, 1 epoch")
print("  (Validates full pipeline end-to-end)")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results = train_fever_nst("configs/fever_gold_smoke.yaml",
                          config_overrides=SMOKE_OVERRIDES)
elapsed = time.time() - t0

print("\n" + "=" * 60)
print(f"SMOKE TEST RESULTS ({elapsed:.0f}s):")
print("=" * 60)
if isinstance(results, dict):
    dev = results.get("dev", {})
    print(f"  dev_accuracy : {dev.get('accuracy', 'N/A')}")
    print(f"  dev_ece      : {dev.get('ece', 'N/A')}")
    print(f"  nan_abort    : {results.get('nan_abort', False)}")
    if results.get("nan_abort", False):
        print("\n  SMOKE TEST FAILED -- NaN detected!")
    else:
        print("\n  Smoke test PASSED -- pipeline working correctly.")

  SMOKE TEST: DeBERTa-v3-base, 200 train, 100 dev, 1 epoch



train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 200 examples (105 with evidence text, 95 without)
fever_dataset |   dev: 100 examples (47 with evidence text, 53 without)
fever_dataset |   train hash: 398028c26c5ec3e7
fever_dataset |   dev hash: e3b09c2bfa26d269
train_fever | Using GOLD EVIDENCE mode (Setting A

  FEVER Dataset Statistics

  train: 200 examples
    With gold evidence: 105 (52.5%)
    Label distribution:
      SUPPORTS                129  (64.5%)
      REFUTES                  16  (8.0%)
      NOT ENOUGH INFO          55  (27.5%)
    Split hash: 398028c26c5ec3e7

  dev: 100 examples
    With gold evidence: 47 (47.0%)
    Label distribution:
      SUPPORTS                 46  (46.0%)
      REFUTES                  30  (30.0%)
      NOT ENOUGH INFO          24  (24.0%)
    Split hash: e3b09c2bfa26d269


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

  Epoch 1/1: loss=1.1638 constraint=0.0000 | dev_acc=0.3300 ECE=0.0340

────────────────────────────────────────
  Post-hoc temperature scaling (dev set)
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


  Learned temperature: T = 2.3206

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
train_fever | Report saved to outputs_fever_gold_smoke/report.json


  Label Accuracy (GOLD evidence): 0.3300
  ECE: 0.0340
  Brier: 0.6808
    SUPPORTS: acc=0.0000 (n=46)
    REFUTES: acc=0.3000 (n=30)
    NOT ENOUGH INFO: acc=1.0000 (n=24)

  Training complete in 8.5s
  Best dev accuracy: 0.3300
  Output: outputs_fever_gold_smoke

SMOKE TEST RESULTS (72s):
  dev_accuracy : 0.33
  dev_ece      : 0.033954
  nan_abort    : False

 Smoke test passed! Pipeline working correctly.


In [6]:
# ============================================================
# Cell 4: Data Leakage Check — verify split integrity BEFORE training
# ============================================================
import os, sys, time

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

print("=" * 60)
print("  DATA LEAKAGE CHECK")
print("  Verifying split disjointness, text overlap, constraint independence")
print("=" * 60 + "\n")

from data.fever_dataset import load_fever_splits
from eval.leakage_check import full_leakage_report

splits = load_fever_splits(max_train=5000, max_dev=2000, seed=42)
report = full_leakage_report(splits)
verdict = report["verdict"]

print(f"\n  Verdict: {'CLEAN -- no leakage detected' if verdict['clean'] else 'ISSUES FOUND'}")
if not verdict["clean"]:
    for issue in verdict.get("issues", []):
        print(f"    - {issue}")
    print("\n  WARNING: Fix leakage issues before reporting results!")
else:
    print("  All checks passed:")
    print("    - Train/Dev split disjoint")
    print("    - Train/Dev_test split disjoint")
    print("    - No suspicious text overlap")
    print("    - Constraints do not leak labels")
print()

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NEURAL BASELINE (Setting A: Gold Evidence)
  DeBERTa-v3-base, Full FEVER train (~145K), 3 epochs
  batch=16, grad_accum=2 (eff. batch=32)
  Eval: every 500 steps on 2K dev subset



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M params)
train_fever | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever | Periodic eval uses dev subset: 2000/17998 examples


...

  Learned temperature: T = 1.1094

────────────────────────────────────────
  Final evaluation on dev set
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Label Accuracy (GOLD evidence): 0.8252
  ECE: 0.0286
  Brier: 0.2445
    SUPPORTS: acc=0.8814 (n=6014)
    REFUTES: acc=0.8266 (n=5969)
    NOT ENOUGH INFO: acc=0.7676 (n=6015)

────────────────────────────────────────
  Final evaluation on held-out dev_test
────────────────────────────────────────


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Label Accuracy (GOLD evidence): 0.8255
  ECE: 0.0384
  Brier: 0.2453
    SUPPORTS: acc=0.8788 (n=652)
    REFUTES: acc=0.8364 (n=697)
    NOT ENOUGH INFO: acc=0.7604 (n=651)

  Training complete in 4405.4s
  Best dev accuracy: 0.8430
  Output: outputs_fever_gold_neural

NEURAL BASELINE RESULTS (81.4 min):
  dev accuracy      : 0.8252
  dev ECE           : 0.028639
  dev Brier         : 0.24446
  dev_test accuracy : 0.8255
  dev_test ECE      : 0.03835
  temperature       : 1.1094
  best_dev_acc      : 0.843

Per-label:
    SUPPORTS: acc=0.8814 (n=6014)
    REFUTES: acc=0.8266 (n=5969)
    NOT ENOUGH INFO: acc=0.7676 (n=6015)

Neural baseline done in 81.4 min


In [7]:
# ============================================================
# Cell 5: NEURAL BASELINE — DeBERTa-v3 + LoRA, Full FEVER
# ============================================================
# Strong baseline: DeBERTa-v3-large (A100) or base (T4) with LoRA
# No symbolic constraints -- pure cross-entropy fine-tuning
# Gold evidence (Setting A) for honest comparison
import os, sys, time, logging, json, gc, torch

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect(); torch.cuda.empty_cache()
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

# Choose config based on GPU tier
if USE_LARGE_MODEL:
    baseline_config = "configs/fever_gold_neural_large.yaml"
    model_desc = "DeBERTa-v3-large + LoRA"
else:
    baseline_config = "configs/fever_gold_neural.yaml"
    model_desc = "DeBERTa-v3-base"

_ov = GPU_OVERRIDES.get("train", {})
print("=" * 60)
print(f"  NEURAL BASELINE (Setting A: Gold Evidence)")
print(f"  {model_desc} | Full FEVER train (~145K) | 3 epochs")
print(f"  bs={_ov.get('batch_size','?')}x{_ov.get('grad_accum_steps','?')} | "
      f"{'BF16' if _ov.get('bf16') else 'FP16'} | "
      f"compile={'ON' if _ov.get('torch_compile') else 'OFF'}")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural = train_fever_nst(baseline_config,
                                  config_overrides=GPU_OVERRIDES)
elapsed = time.time() - t0

print("\n" + "=" * 60)
print(f"NEURAL BASELINE RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
dev = results_neural.get("dev", {})
dev_test = results_neural.get("dev_test", {})
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
    print(f"  dev_test ECE      : {dev_test.get('ece', 'N/A')}")
print(f"  temperature       : {results_neural.get('temperature', 'N/A')}")
print(f"  best_dev_acc      : {results_neural.get('best_dev_acc', 'N/A')}")
print(f"\nPer-label:")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label}: acc={stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_neural.json", "w") as f:
    json.dump(results_neural, f, indent=2, default=str)
print(f"\nNeural baseline done in {elapsed/60:.1f} min")

train_fever | Loading FEVER dataset...
fever_dataset | Loading FEVER from HuggingFace datasets...


  NST SOFT CONSTRAINTS (Setting A: Gold Evidence)
  CE + fixed lambda=0.1 x constraint_loss
  5 constraints: date, number, negation, entity, empty



fever_dataset |   Using SQLite wiki cache: /content/nst/data/fever_wiki.db (14363 pages)
fever_dataset |   Wiki page map: 14363 pages loaded
fever_dataset |   train: 145449 examples (77591 with evidence text, 67858 without)
fever_dataset |   dev: 19998 examples (8441 with evidence text, 11557 without)
fever_dataset |   Split labelled_dev into dev (17998) + dev_test (2000)
fever_dataset |   train hash: d5ca82052d828bc1
fever_dataset |   dev hash: 339d321ff17fccd2
fever_dataset |   dev_test hash: fdf5b26ced54b5f9
train_fever | Using GOLD EVIDENCE mode (Setting A)
fever_nli | Loading model: microsoft/deberta-v3-base


  FEVER Dataset Statistics

  train: 145449 examples
    With gold evidence: 77591 (53.3%)
    Label distribution:
      SUPPORTS              80035  (55.0%)
      REFUTES               29775  (20.5%)
      NOT ENOUGH INFO       35639  (24.5%)
    Split hash: d5ca82052d828bc1

  dev: 17998 examples
    With gold evidence: 7581 (42.1%)
    Label distribution:
      SUPPORTS               6014  (33.4%)
      REFUTES                5969  (33.2%)
      NOT ENOUGH INFO        6015  (33.4%)
    Split hash: 339d321ff17fccd2

  dev_test: 2000 examples
    With gold evidence: 860 (43.0%)
    Label distribution:
      SUPPORTS                652  (32.6%)
      REFUTES                 697  (34.9%)
      NOT ENOUGH INFO         651  (32.5%)
    Split hash: fdf5b26ced54b5f9


Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
fever_nli | Model loaded: microsoft/deberta-v3-base (184.4M params)
train_fever | Class weights: [0.6057724952697754, 1.628312349319458, 1.3603917360305786]
train_fever | Periodic eval uses dev subset: 2000/17998 examples


...

  Step 4500: loss=0.3039 λ=0.1000 | dev_acc=0.8415 ECE=0.0185


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

  Epoch 1/3: loss=0.5839 constraint=0.4076


Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

: 

In [1]:
# ============================================================
# Cell 6: NST-VERI (FLAGSHIP) — Full Neurosymbolic Method
# ============================================================
# The flagship method: Verification-Enhanced Reasoning Integration
#
# Architecture:
#   DeBERTa-v3 backbone + LoRA
#   + 6 verification heads (auxiliary supervision from constraints)
#   + Zero-init residual correction conditioned on verification
#   + Supervised contrastive loss on high-confidence constraint examples
#   + Per-sample adaptive lambda via uncertainty-aware gating
#
# Training phases:
#   Phase 1 (Epoch 0):     Pure NLI + auxiliary verification tasks
#   Phase 2 (Epoch 1):     Add contrastive loss (gamma=0.1)
#   Phase 3 (Epochs 2-4):  Add constraints with lambda warmup to 0.3
#
# 6 probabilistic constraints (v2):
#   NumericalConstraint, NegationConstraint, EntityOverlapConstraint,
#   EvidenceSufficiencyConstraint, TemporalConstraint, HedgeModalityConstraint
import os, sys, time, logging, json, gc, torch

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                        "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

gc.collect(); torch.cuda.empty_cache()
logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

_ov = VERI_OVERRIDES.get("train", {})
_model = VERI_OVERRIDES.get("model", {}).get("name", "deberta-v3")
print("=" * 60)
print("  NST-VERI: Verification-Enhanced Reasoning Integration")
print("  (FLAGSHIP NEUROSYMBOLIC METHOD)")
print("=" * 60)
print(f"  Model     : {_model}")
print(f"  LoRA      : r={VERI_OVERRIDES.get('model', {}).get('lora_rank', '?')}")
print(f"  Batch     : {_ov.get('batch_size','?')}x{_ov.get('grad_accum_steps','?')}")
print(f"  Precision : {'BF16' if _ov.get('bf16') else 'FP16'}")
print(f"  Phases    : 3 (NLI+aux -> +contrastive -> +constraints)")
print(f"  Constraints: 6 probabilistic (v2)")
print(f"  Lambda    : adaptive, max=0.3")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_nst_veri import train_nst_veri
results_veri = train_nst_veri("configs/fever_gold_nst_veri.yaml",
                               config_overrides=VERI_OVERRIDES)
elapsed = time.time() - t0

print("\n" + "=" * 60)
print(f"NST-VERI RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
dev = results_veri.get("dev", {})
dev_test = results_veri.get("dev_test", {})
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
    print(f"  dev_test ECE      : {dev_test.get('ece', 'N/A')}")
print(f"  temperature       : {results_veri.get('temperature', 'N/A')}")
print(f"  best_dev_acc      : {results_veri.get('best_dev_acc', 'N/A')}")
print(f"\nPer-label:")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label}: acc={stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

with open("results_veri.json", "w") as f:
    json.dump(results_veri, f, indent=2, default=str)
print(f"\nNST-VERI done in {elapsed/60:.1f} min")

FileNotFoundError: [Errno 2] No such file or directory: '/content/nst'

In [ ]:
# ============================================================
# Cell 7: Legacy Modes (Optional) — Soft, CEGIS, Gated
# ============================================================
# Run these only if you want to compare all modes.
# The primary comparison is: Neural Baseline vs NST-VERI (cells 5+6)
import os, sys, time, logging, json, gc, torch

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

RUN_LEGACY_MODES = False  # <-- Set True to run soft/cegis/gated comparison

if RUN_LEGACY_MODES:
    legacy_configs = {
        "soft":  "configs/fever_gold_nst_soft.yaml",
        "cegis": "configs/fever_gold_nst_cegis.yaml",
        "gated": "configs/fever_gold_nst_gated.yaml",
    }

    legacy_results = {}
    for mode_name, cfg_path in legacy_configs.items():
        if not os.path.exists(cfg_path):
            print(f"  Skipping {mode_name}: config not found ({cfg_path})")
            continue

        # Clear modules
        for mod in list(sys.modules.keys()):
            if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.",
                                                "logic.", "symbolic.", "retrieval."]):
                del sys.modules[mod]

        gc.collect(); torch.cuda.empty_cache()
        logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

        print(f"\n{'='*60}")
        print(f"  LEGACY MODE: {mode_name.upper()} (v1 constraints)")
        print(f"{'='*60}\n")

        t0 = time.time()
        from training.train_fever_nst import train_fever_nst
        result = train_fever_nst(cfg_path, config_overrides=GPU_OVERRIDES)
        elapsed = time.time() - t0

        legacy_results[mode_name] = result
        dev = result.get("dev", {})
        print(f"\n  {mode_name.upper()}: dev_acc={dev.get('accuracy', 'N/A')}, "
              f"ECE={dev.get('ece', 'N/A')}, time={elapsed/60:.1f}min")

        with open(f"results_{mode_name}.json", "w") as f:
            json.dump(result, f, indent=2, default=str)

    print("\nLegacy modes complete.")
else:
    print("Legacy modes skipped. Set RUN_LEGACY_MODES = True in this cell to run.")

In [ ]:
# ============================================================
# Cell 8: Results Summary — Comparison Table + Statistical Tests
# ============================================================
import json, os, glob
import numpy as np

os.chdir("/content/nst")

print("=" * 72)
print("  FEVER RESULTS — Setting A: Gold Evidence")
print("  Honest evaluation, no data leakage, held-out dev_test")
print("=" * 72)
print()

# Load all results
results = {}
name_map = {
    "results_neural.json": "Neural Baseline",
    "results_veri.json": "NST-VERI (flagship)",
    "results_soft.json": "NST-Soft (v1)",
    "results_cegis.json": "NST-CEGIS (v1)",
    "results_gated.json": "NST-ECCG (v1)",
}

for fname, name in name_map.items():
    if os.path.exists(fname):
        with open(fname) as f:
            results[name] = json.load(f)

if not results:
    print("No results found! Run cells 5-7 first.")
else:
    # ── Main table ──
    print(f"  {'Method':<22} {'Dev Acc':<10} {'ECE':<10} {'Brier':<10} "
          f"{'DevTest':<10} {'Temp':<8} {'Time':<8}")
    print("  " + "-" * 78)

    for name, r in results.items():
        dev  = r.get("dev", {})
        dt   = r.get("dev_test", {})
        temp = r.get("temperature", 0)
        elapsed = r.get("elapsed_s", 0)
        d_acc = dev.get("accuracy", 0)
        d_ece = dev.get("ece", 0)
        d_bri = dev.get("brier", 0)
        dt_acc = dt.get("accuracy", 0) if dt else 0
        dt_s   = f"{dt_acc:<10.4f}" if dt else "    --    "
        temp_s = f"{temp:<8.4f}" if isinstance(temp, (int, float)) else f"{str(temp):<8}"
        t_s    = f"{elapsed/60:<8.1f}" if elapsed else "  --  "
        print(f"  {name:<22} {d_acc:<10.4f} {d_ece:<10.4f} {d_bri:<10.4f} "
              f"{dt_s} {temp_s} {t_s}")

    print()

    # ── Per-label breakdown for best model ──
    best_name = max(results, key=lambda n: results[n].get("dev", {}).get("accuracy", 0))
    best_acc  = results[best_name].get("dev", {}).get("accuracy", 0)
    print(f"  BEST DEV ACCURACY: {best_name} ({best_acc:.4f})")

    best_dev = results[best_name].get("dev", {})
    print(f"\n  Per-label breakdown ({best_name}):")
    for label, stats in best_dev.get("per_label", {}).items():
        prec = stats.get("precision", 0)
        prec_s = f" prec={prec:.4f}" if isinstance(prec, float) else ""
        print(f"    {label}: acc={stats.get('accuracy', 0):.4f}{prec_s} "
              f"(n={stats.get('count', 0)})")

    # ── DevTest (held-out, final metric) ──
    print(f"\n  DevTest (held-out, FINAL metric):")
    for name, r in results.items():
        dt = r.get("dev_test", {})
        if dt:
            print(f"    {name}: acc={dt.get('accuracy', 0):.4f} "
                  f"ECE={dt.get('ece', 0):.4f}")

    # ── Delta analysis (if both neural and veri exist) ──
    if "Neural Baseline" in results and "NST-VERI (flagship)" in results:
        n_dev = results["Neural Baseline"].get("dev", {}).get("accuracy", 0)
        v_dev = results["NST-VERI (flagship)"].get("dev", {}).get("accuracy", 0)
        delta = v_dev - n_dev
        print(f"\n  NST-VERI vs Neural Baseline:")
        print(f"    Dev accuracy delta : {delta:+.4f} ({delta*100:+.2f}pp)")
        n_dt = results["Neural Baseline"].get("dev_test", {}).get("accuracy", 0)
        v_dt = results["NST-VERI (flagship)"].get("dev_test", {}).get("accuracy", 0)
        if n_dt and v_dt:
            delta_dt = v_dt - n_dt
            print(f"    DevTest acc delta  : {delta_dt:+.4f} ({delta_dt*100:+.2f}pp)")
        n_ece = results["Neural Baseline"].get("dev", {}).get("ece", 0)
        v_ece = results["NST-VERI (flagship)"].get("dev", {}).get("ece", 0)
        print(f"    ECE improvement    : {n_ece - v_ece:+.4f}")

    # ── Confusion matrix for best model ──
    print(f"\n  Confusion matrix ({best_name}, dev set):")
    cm = best_dev.get("confusion", {})
    if cm:
        labels = list(cm.keys())
        header = "  " + " " * 22 + "  ".join(f"{l[:8]:>10}" for l in labels)
        print(header)
        for gold in labels:
            row = f"  {gold:<20}"
            for pred in labels:
                row += f"  {cm.get(gold, {}).get(pred, 0):>10}"
            print(row)

    print(f"\n{'='*72}")
    print("  Key:")
    print("  - Dev      : Main development set (used for tuning)")
    print("  - DevTest  : Held-out 10% of dev (NEVER tuned on, final metric)")
    print("  - ECE      : Expected Calibration Error (lower = better)")
    print("  - Brier    : Brier score (lower = better)")
    print("  - Temp     : Post-hoc temperature scaling fit on dev")
    print(f"{'='*72}")

print("\nAll experiments complete!")
print("Full reports in outputs_fever_gold_*/report.json")
print("\nRemember: Runtime > Disconnect and delete runtime (save credits!)")

In [ ]:
# ============================================================
# Cell 9: (Optional) Save Results to Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import shutil, datetime, os, glob

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")

# Collect all FEVER outputs
os.makedirs("all_fever_outputs", exist_ok=True)
for d in sorted(glob.glob("outputs_fever_*")):
    dst = os.path.join("all_fever_outputs", os.path.basename(d))
    if not os.path.exists(dst):
        shutil.copytree(d, dst)

# Also copy result JSONs
for f in glob.glob("results_*.json"):
    shutil.copy(f, "all_fever_outputs/")

archive = shutil.make_archive(f"nst_fever_results_{timestamp}", "zip", "all_fever_outputs")
print(f"Created: {archive}")

dst_dir = "/content/drive/MyDrive/"
shutil.copy(archive, dst_dir)
print(f"Saved to Drive: {dst_dir}{os.path.basename(archive)}")
print("\nAll results saved to Google Drive")